# KADMON Optuna — K-Means + Partial OT

Le classement utilise `global_distance_mm`. Pour chaque bundle, l'objectif Optuna minimise le rapport entre la distance intra-identité et la distance moyenne inter-identité.


## 1. Imports et configuration


In [1]:
from pathlib import Path
import os
import sys
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed


NOTEBOOK_DIR = Path.cwd().resolve()
KADMON_ROOT = (NOTEBOOK_DIR / "../..").resolve()
BUNDLES_DIR = NOTEBOOK_DIR.parent / "bundles"
STUDY_PATH = NOTEBOOK_DIR / "studies" / "optuna_reid.sqlite3"
if str(KADMON_ROOT) not in sys.path:
    sys.path.insert(0, str(KADMON_ROOT))

from kadmon.comparison import compare_bundles
from kadmon.io import BundleCollection
from kadmon.protocol import HCP_REID_PROTOCOL
from kadmon.reid import aggregate_reid_metrics, bundle_reid_metrics, set_trial_metrics
from kadmon.selection import select_best_reid_trial
from kadmon.optimization import StudyRunPolicy, create_reid_study, run_until_complete

optuna.logging.set_verbosity(optuna.logging.WARNING)

EXPERIMENT_NAME = "kmeans_partial"
COMPRESSION = "kmeans"
TRANSPORT = "partial"
N_TRIALS = 80


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## 2. Données

Les bundles sont chargés depuis le dossier frère `../bundles`.


In [2]:
REFERENCE_SUBJECT = HCP_REID_PROTOCOL.reference_subject
INTRA_IDENTITY_SUBJECT = HCP_REID_PROTOCOL.intra_identity_subject
COMPARISON_SUBJECTS = HCP_REID_PROTOCOL.comparison_subjects
SUBJECTS = HCP_REID_PROTOCOL.subjects
N_POINTS = HCP_REID_PROTOCOL.n_points
SEED = 42
BUNDLES_TO_RUN = None  # tuple de noms exacts pour un essai court, sinon None
N_JOBS_BUNDLES = min(8, os.cpu_count() or 1)

bundles = BundleCollection(
    BUNDLES_DIR, SUBJECTS, n_points=N_POINTS, selected=BUNDLES_TO_RUN
)
SUBJECT_FILES = bundles.files
bundle_names = bundles.names
bundle_cache = bundles.cache
# Le cache inclut aussi le type et tous les paramètres de compression via kadmon.
compression_cache = {}


load_bundle = bundles.load


print(f"Données : {BUNDLES_DIR}")
print("Protocole : 1 comparaison intra-identité et 9 comparaisons inter-identité par bundle")
print(f"Bundles communs : {len(bundle_names)}")
display(pd.DataFrame({"bundle": bundle_names}))


Données : /home/colin/Tractographie/KADMON/notebooks/bundles
Protocole : 1 comparaison intra-identité et 9 comparaisons inter-identité par bundle
Bundles communs : 31


,bundle
0,tractosearch_nn_8_0mm_all_AF_L_m
1,tractosearch_nn_8_0mm_all_AF_R_m
2,tractosearch_nn_8_0mm_all_CC_1_m
3,tractosearch_nn_8_0mm_all_CC_2a_m
4,tractosearch_nn_8_0mm_all_CC_2b_m
5,tractosearch_nn_8_0mm_all_CC_3_m
6,tractosearch_nn_8_0mm_all_CC_4_m
7,tractosearch_nn_8_0mm_all_CC_5_m
8,tractosearch_nn_8_0mm_all_CC_6_m
9,tractosearch_nn_8_0mm_all_CC_7_m


## 3. Espace de recherche

- `n_clusters` : 25 à 300, par pas de 5
- `mass` : 0,50 à 1,00, par pas de 0,01

`n_points=12`, la seed, `max_iter=300` et `tol=1e-4` restent fixes.


In [3]:
def sample_parameters(trial):
    return (
        {
            "n_clusters": trial.suggest_int("n_clusters", 25, 300, step=5),
            "max_iter": 300,
            "tol": 1e-4,
            "seed": SEED,
        },
        {"mass": trial.suggest_float("mass", 0.50, 1.00, step=0.01)},
    )


## 4. Objectif Optuna

La compression, MDF, le transport et les statistiques sont calculés par `compare_bundles()`.


In [4]:
def evaluate_bundle(bundle_name, compression_parameters, transport_parameters):
    source = load_bundle(REFERENCE_SUBJECT, bundle_name)
    pair_compression_parameters = dict(compression_parameters)

    minimum_size = min(
        len(load_bundle(subject, bundle_name)) for subject in SUBJECTS
    )
    pair_compression_parameters["n_clusters"] = min(
        int(compression_parameters["n_clusters"]), minimum_size
    )

    pair_metrics = []
    for candidate_subject in COMPARISON_SUBJECTS:
        target = load_bundle(candidate_subject, bundle_name)
        result = compare_bundles(
            source,
            target,
            compression=COMPRESSION,
            transport=TRANSPORT,
            compression_parameters=pair_compression_parameters,
            transport_parameters=transport_parameters,
            compression_cache=compression_cache,
            source_compression_key=(REFERENCE_SUBJECT, bundle_name),
            target_compression_key=(candidate_subject, bundle_name),
        )
        metrics = result["metrics"]
        pair_metrics.append({
            "global_distance_mm": float(metrics["global_distance_mm"]),
            "mean_displacement_mm": float(metrics["mean_mm"]),
            "transported_mass": float(metrics["transported_mass"]),
            "source_n_representatives": int(metrics["source_n_representatives"]),
            "target_n_representatives": int(metrics["target_n_representatives"]),
        })

    # La compression de la référence doit être reproductible pour tous les candidats.
    if len({row["source_n_representatives"] for row in pair_metrics}) != 1:
        raise RuntimeError(f"Compression source non reproductible pour {bundle_name}.")

    return bundle_reid_metrics(
        bundle_name, [row["global_distance_mm"] for row in pair_metrics],
        comparison_subjects=COMPARISON_SUBJECTS,
        intra_identity_subject=INTRA_IDENTITY_SUBJECT,
        mean_displacement_mm=np.mean([row["mean_displacement_mm"] for row in pair_metrics]),
        mean_transported_mass=np.mean([row["transported_mass"] for row in pair_metrics]),
        mean_n_representatives=np.mean([value for row in pair_metrics for value in (row["source_n_representatives"], row["target_n_representatives"])]),
        extra={"n_clusters_effective": pair_compression_parameters["n_clusters"]},
    )


def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    tasks = (
        delayed(evaluate_bundle)(
            bundle_name, compression_parameters, transport_parameters
        )
        for bundle_name in bundle_names
    )
    results = (
        [
            evaluate_bundle(
                bundle_name, compression_parameters, transport_parameters
            )
            for bundle_name in bundle_names
        ]
        if N_JOBS_BUNDLES == 1
        else Parallel(
            n_jobs=min(N_JOBS_BUNDLES, len(bundle_names)), backend="threading"
        )(tasks)
    )
    bundle_metrics = pd.DataFrame(results)

    aggregates = aggregate_reid_metrics(
        bundle_metrics, n_comparison_subjects=len(COMPARISON_SUBJECTS),
        elapsed_s=perf_counter() - started, extra={
            "n_clusters_effective_min": int(bundle_metrics["n_clusters_effective"].min()),
            "n_clusters_effective_max": int(bundle_metrics["n_clusters_effective"].max()),
        },
    )
    set_trial_metrics(trial, aggregates)
    return aggregates["mean_intra_inter_ratio"]


## 5. Optimisation

La base SQLite est l'unique sortie automatique de l'étude.


In [5]:
study = create_reid_study(STUDY_PATH, EXPERIMENT_NAME, seed=SEED)
run_until_complete(study, objective, StudyRunPolicy(N_TRIALS))

print(f"Base SQLite : {STUDY_PATH}")


Étude : 80/80 essais COMPLETE; cible restante=0.
Base SQLite : /home/colin/Tractographie/KADMON/notebooks/optuna/studies/optuna_reid.sqlite3


## 6. Analyse des essais observés

Analyse des essais `COMPLETE` déjà présents dans `study` ; le score intra/inter-identité est à minimiser.

In [6]:
df = study.trials_dataframe()[["number", "value", "params_mass", "params_n_clusters", "user_attrs_intra_identity_top1_accuracy"]]
df = df.dropna(subset=["value"]).rename(columns={
    "number": "trial", "value": "score",
    "params_mass": "mass", "params_n_clusters": "n_clusters",
    "user_attrs_intra_identity_top1_accuracy": "reid_accuracy",
})
best_score = df["score"].min()

display(df.sort_values("score")[["trial", "score", "mass", "n_clusters", "reid_accuracy"]]
        .style.format({"score": "{:.6f}", "mass": "{:.2f}", "reid_accuracy": "{:.0%}"}))

reid_best = select_best_reid_trial(study.trials)
display(pd.Series({"experiment": EXPERIMENT_NAME, "trial": reid_best.number,
                   "selection": "Top-1, rang, ratio, couverture",
                   "mean_intra_inter_ratio": reid_best.value, **reid_best.params,
                   **reid_best.user_attrs}, name="meilleur essai RE-ID").to_frame())
reid = reid_best.user_attrs
if "reid_valid_bundles" in reid:
    print("RE-ID validée :", reid["reid_valid_bundles"])
    print("RE-ID échouée :", reid["reid_failed_bundles"])

rows = []
for threshold in (1, 2, 5):
    candidates = df[df["score"] <= best_score * (1 + threshold / 100)]
    row = candidates.sort_values(["mass", "score"], ascending=[False, True]).iloc[0]
    rows.append([threshold, int(row.trial), row.score, 100 * (row.score / best_score - 1), row.mass, int(row.n_clusters), row.reid_accuracy])
tradeoff = pd.DataFrame(rows, columns=["seuil (%)", "trial", "score", "écart relatif (%)", "mass", "n_clusters", "reid_accuracy"])
display(tradeoff[["seuil (%)", "trial", "score", "écart relatif (%)", "mass", "n_clusters", "reid_accuracy"]]
        .style.format({"score": "{:.6f}", "écart relatif (%)": "{:.2f}", "mass": "{:.2f}", "reid_accuracy": "{:.0%}"}))

optuna.visualization.plot_contour(study, params=["mass", "n_clusters"]).show()

,trial,score,mass,n_clusters,reid_accuracy
79,79,0.334239,0.54,40,100%
59,59,0.335405,0.51,65,100%
86,86,0.336589,0.50,45,100%
48,48,0.340457,0.59,40,100%
39,39,0.341698,0.54,45,100%
9,9,0.342628,0.57,65,100%
46,46,0.343802,0.58,65,100%
53,53,0.343802,0.58,65,100%
73,73,0.344309,0.56,45,100%
72,72,0.344309,0.56,45,100%


,meilleur essai RE-ID
experiment,kmeans_partial
trial,79
selection,"Top-1, rang, ratio, couverture"
mean_intra_inter_ratio,0.334239
n_clusters,40
mass,0.54
elapsed_s,0.583325
intra_identity_top1_accuracy,1.0
mean_displacement_mm,3.82049
mean_inter_identity_distance_mm,4.292829


,seuil (%),trial,score,écart relatif (%),mass,n_clusters,reid_accuracy
0,1,79,0.334239,0.00,0.54,40,100%
1,2,48,0.340457,1.86,0.59,40,100%
2,5,44,0.346274,3.60,0.63,40,100%


## 7. Interprétation des résultats

À moins de 1 % du meilleur score, aucune masse supérieure à 0,54 n'est observée. Le trial 48 atteint mass=0,59 pour une dégradation relative de 1,86 %, tandis que le trial 44 atteint mass=0,63 pour une dégradation de 3,60 %.

Selon la règle RE-ID commune, le trial 79 est le meilleur classement : Top-1 100 %, rang moyen 1,00, `n_clusters=40` et `mass=0,54`. Pour l'usage anatomique de KADMON, le **trial 44 est retenu par défaut**, avec `mass=0,63` pour une dégradation relative du ratio limitée à 3,60 % et une couverture supérieure.



- En **RE-ID**, un faible nombre de clusters et une faible masse Partial OT semblent améliorer les performances de ré-identification.
- En **analyse anatomique**, une faible masse peut négliger une partie de la géométrie du faisceau.
- Le **trial 44** peut donc être privilégié pour les analyses où la conservation d'une plus grande partie du faisceau est importante.